<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# MSLE and gradients

Today we're implementing Gradient Descent by hand.
We'll also look at the Mean Squared Logarithmic Error (MSLE).

In [5]:
%matplotlib inline

from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from tqdm.auto import tqdm

NDArray = np.ndarray[Any, Any]

np.set_printoptions(precision=4, suppress=True)
np.random.seed(357)

## Loading and converting data

Same data as before, cleaned up. We'll use NumPy arrays instead of Pandas DataFrames to simplify the implementation.

In [68]:
!wget --no-verbose -O lab1_apartments_train_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv
!wget --no-verbose -O lab1_apartments_test_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv
!head lab1_apartments_train_clean.csv lab1_apartments_test_clean.csv

2026-10-09 09:44:09 URL:https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv [6187/6187] -> "lab1_apartments_train_clean.csv" [1]
2026-10-09 09:44:10 URL:https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv [6223/6223] -> "lab1_apartments_test_clean.csv" [1]
==> lab1_apartments_train_clean.csv <==
area,district,bedrooms,bathrooms,build_year,parking,price
104,mokotowo,2,2,1940,1,780094
43,ochotowo,1,1,1970,1,346912
128,grodziskowo,3,2,1916,1,523466
112,mokotowo,3,2,1920,1,830965
149,mokotowo,3,3,1977,0,1090479
80,ochotowo,2,2,1937,0,599060
58,ochotowo,2,1,1922,0,463639
23,ochotowo,1,1,1929,0,166785
40,mokotowo,1,1,1973,0,318849

==> lab1_apartments_test_clean.csv <==
area,district,bedrooms,bathrooms,build_year,parking,price
71,wolowo,2,2,1912,1,322227
45,mokotowo,1,1,1938,0,295878
38,mokotowo,1,1,1999,1,306530
70,ochotowo,2,2,1980,1,553641
136,mokotowo,3,2,1939,1,985348
128,wolowo,3,2,1983,1,695726
23,grodziskowo,1,1,1975,0,99751
117,mokotowo,3,2,1942,0,

In [69]:
def load_dataset(path: str) -> tuple[NDArray, NDArray]:
    """
    Returns (x, y) where:
    - x: input features, shape (n_apartments, n_features)
    - y: prices, shape (n_apartments,)
    """
    data = pd.read_csv(path)
    y = data["price"].to_numpy()
    x = data.loc[:, data.columns != "price"].to_numpy()
    return x, y

x_train, y_train = load_dataset("lab1_apartments_train_clean.csv")
x_test, y_test = load_dataset("lab1_apartments_test_clean.csv")

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

print(x_test[0, :])

(200, 6) (200,)
(200, 6) (200,)
[71 'wolowo' 2 2 1912 1]


In [70]:
better_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False,
    drop='first'
)

better_encoder.fit(x_train[:, [1]])

x_train_district = better_encoder.transform(x_train[:, [1]])
x_test_district = better_encoder.transform(x_test[:, [1]])

x_train_encoded = np.concatenate(
    [x_train[:, [0]], x_train_district, x_train[:, 2:]],
    axis=1
).astype(np.float64)

x_test_encoded = np.concatenate(
    [x_test[:, [0]], x_test_district, x_test[:, 2:]],
    axis=1
).astype(np.float64)

print(x_train_encoded.shape)
print(x_test_encoded.shape)
print(x_train_encoded.dtype)
print(x_test_encoded.dtype)

(200, 8)
(200, 8)
float64
float64


## The loss, constant models

The range of prices is quite large.

In [55]:
print(np.min(y_train), np.max(y_train), np.mean(y_train))

102572 1102309 507919.49


When predicting them, relative errors are arguably more important than absolute errors.<br>
That is, mistaking 1M vs 2M is arguably similar to mistaking 11M vs 22M, more so than mistaking 11M vs 12M.

Therefore, let's assume that our predictions should minimize the so-called *mean squared logarithmic error*:
$$
MSLE(\bar{y}, \bar{p}) = \frac{1}{n} \sum_{i=1}^n (\log(1+y_i) - \log(1+p_i))^2,
$$
where $y_i$ is the ground truth (the target), and $p_i$ is our prediction.

Let's implement the loss function first.

In [56]:
def mse(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    return np.mean((ys - ps) * (ys - ps))

In [57]:
def msle(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    return np.mean((np.log(1 + ys) - np.log(1 + ps)) * (np.log(1 + ys) - np.log(1 + ps)))

The simplest model is predicting the same constant for each instance. Test your implementation of msle against outputing the mean price.

In [58]:
###################################################
# TODO: Compute msle for outputing the mean price #
###################################################
msle_against_mean_price = msle(y_test, np.mean(y_train) * np.ones(y_test.shape))
print(msle_against_mean_price)

0.4284115392580848


To get a feel of whether this error is high or low, we can apply exp and sqrt to get some kind of mean (`exp of quadratic mean of log`) in interpretable units.<br>
(For context, observe that `exp of arithmetic mean of log` is the geometric mean).

In [59]:
interpretable_error = np.exp(np.sqrt(msle_against_mean_price))
print(interpretable_error)

1.9242408798532662


So we are roughly 1.92× off the actual y_test when we predict that constant value all the time, on average.

Recall that outputing the **mean** minimizes $MSE$. However, we're now dealing with $MSLE$.

Find a constant that should result in the lowest $MSLE$.

In [ ]:
#############################################
# TODO: Find this constant and compute msle #
#############################################
minimise =


## Linear regression (standard)

Now, let's implement training of a standard linear regression model via gradient descent.

In [67]:
def train(
    x: NDArray, y: NDArray, alpha: float = 1e-7, n_iterations: int = 100000
) -> tuple[NDArray, np.float64]:
    """Linear regression (which optimizes MSE). Returns (weights, bias)."""

    # B is batch size (number of observations).
    # D is number of input features.
    B, D = x.shape
    assert y.shape == (B,)

    weights = (1/np.sqrt(D)) * np.random.uniform(low=-1, high=1, size=(D,))
    bias = np.mean(y)

    for _ in range(n_iterations):
        pred_train = x @ weights + bias
        error = pred_train - y

        grad_weights = (2 / B) * (x.T @ error)
        grad_bias = (2 / B) * np.sum(error)

        weights -= alpha * grad_weights
        bias -= alpha * grad_bias

    return weights, bias

weights, bias = train(x_train_encoded, y_train)
preds_test = x_test_encoded @ weights + bias  # shapes (B, D) @ (D,) + ()

print("test MSLE:", msle(y_test, preds_test))
# Should be much better than the constant model.

test MSLE: 0.07886485387763742


## Linear regression (MSLE)

Note that the loss function that the algorithms optimizes (i.e $MSE$) differs from $MSLE$. We've already seen that this may result in a suboptimal solution.

How can you change the setting so that we optimze $MSLE$ instead?

Hint:
<sub><sup><sub><sup><sub><sup>
Be lazy. We don't want to change the algorithm.
Use the chain rule and previous computations to get formulas for the gradient.
</sup></sub></sup></sub></sup></sub>

In [ ]:
def train_msle(
    x: NDArray, y: NDArray, alpha: float = 1e+4, n_iterations: int = 50000
) -> tuple[NDArray, NDArray]:
    """Gradient descent for MSLE."""

    #############################################
    # TODO: Optimize msle and compare the error #
    #############################################

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)


    return weights, bias


weights, bias = train_msle(x_train, y_train)
preds_test = x_test @ weights + bias
print("test MSLE: ", msle(y_test, preds_test))
# Train MSLE should be better. Test MSLE not necessarily.